In [27]:
# Cellule 1 — Imports avec rechargement forcé
import sys
import os

# Vider le cache des modules src
mods_to_remove = [key for key in sys.modules if key.startswith("src") or key.startswith("config")]
for mod in mods_to_remove:
    del sys.modules[mod]
print("🧹 Cache modules vidé")

sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
print("✅ libs de base OK")

from dotenv import load_dotenv
load_dotenv()
print("✅ dotenv OK")

from src.db import fetch_all
print("✅ src.db OK")

from src.features import build_run_features
print("✅ src.features OK")

from src.scoring import compute_scores, ml_enrichment, print_run_report
print("✅ src.scoring OK")

from config.scoring_config import SCORING_CONFIG
print("✅ scoring_config OK")

print("\n🚀 Tous les imports sont prêts !")




🧹 Cache modules vidé
✅ libs de base OK
✅ dotenv OK
✅ src.db OK
✅ src.features OK
✅ src.scoring OK
✅ scoring_config OK

🚀 Tous les imports sont prêts !


In [28]:
print("📥 Chargement des données...")

df_runs      = fetch_all("Run")
df_stages    = fetch_all("Stage")
df_rooms     = fetch_all("Room")
df_pstates   = fetch_all("PlayerState")
df_inventory = fetch_all("Inventory")
df_passive   = fetch_all("PassiveItem")
df_active    = fetch_all("ActiveItem")
df_trinkets  = fetch_all("Trinket")
df_pactive   = fetch_all("PlayerStateActive")
df_ptrinket  = fetch_all("PlayerStateTrinket")
df_rmonster  = fetch_all("RoomMonster")
df_rboss     = fetch_all("RoomBoss")

print(f"✅ {len(df_runs)} runs chargées")


📥 Chargement des données...
✅ 20 runs chargées


In [29]:
print("⚙️  Construction des features...")

df_features = build_run_features(
    df_runs, df_stages, df_rooms, df_pstates,
    df_inventory, df_passive, df_pactive,
    df_ptrinket, df_rmonster, df_rboss
)

print(f"✅ {df_features.shape[1] - 1} features pour {len(df_features)} runs")
df_features.head()


⚙️  Construction des features...
✅ 37 features pour 20 runs


,run_id,victory,difficulty,character,duration,nb_stages,nb_curses,nb_rooms,nb_rooms_cleared,clear_rate,...,delta_move_speed,dps_proxy,nb_passive_items,avg_passive_quality,total_passive_quality,nb_active_items,avg_active_quality,nb_trinkets,avg_trinket_quality,rooms_per_frame
0,2,0,0,0,13522,2,32,11,10,0.909091,...,0.0,0.280000,3,1.945946,144.0,1,0.0,0,0.0,0.000740
1,3,0,0,0,8854,1,0,10,9,0.900000,...,0.0,0.350000,2,0.000000,0.0,1,0.0,0,0.0,0.001016
2,4,0,0,0,17990,2,0,12,11,0.916667,...,0.0,0.350000,2,0.590909,13.0,1,0.0,0,0.0,0.000611
3,5,0,0,0,6484,1,0,6,5,0.833333,...,0.0,0.454083,2,0.333333,3.0,1,0.0,0,0.0,0.000771
4,6,0,0,0,2468,1,0,1,0,0.000000,...,0.0,0.350000,0,0.000000,0.0,0,0.0,0,0.0,0.000000


In [30]:
print("🏆 Calcul des scores...")
df_scored = compute_scores(df_features, SCORING_CONFIG)

cols = ["run_id", "score", "grade", "rank", "victory",
        "nb_stages", "dps_proxy", "total_damage_taken",
        "nb_passive_items", "avg_passive_quality"]
cols = [c for c in cols if c in df_scored.columns]

df_scored[cols].head(10)


🏆 Calcul des scores...


,run_id,score,grade,rank,victory,nb_stages,dps_proxy,total_damage_taken,nb_passive_items,avg_passive_quality
19,21,59.14,C,1,0,8,0.546875,4.0,12,1.562166
12,14,39.03,D,2,0,3,0.585725,5.0,5,1.411765
1,3,32.41,D,3,0,1,0.350000,1.0,2,0.000000
14,16,31.59,D,4,0,2,0.350000,3.0,1,1.000000
3,5,29.88,D,5,0,1,0.454083,2.0,2,0.333333
13,15,29.10,D,6,0,1,0.421902,5.0,4,2.428571
16,18,28.58,D,7,0,1,0.350000,1.0,0,0.000000
11,13,28.54,D,8,0,1,0.350000,2.0,1,0.000000
9,11,28.33,D,9,0,1,0.421902,3.0,1,2.000000
7,9,28.20,D,10,0,1,0.350000,0.0,0,0.000000


In [31]:
df_scored = ml_enrichment(df_scored, SCORING_CONFIG)
df_scored[["run_id", "score", "grade", "cluster_label", "anomaly_label"]].head(10)



🔍 Répartition des clusters :
               count   mean    min    max
cluster_label                            
⭐ Solide           1  39.03  39.03  39.03
🏆 Elite            1  59.14  59.14  59.14
💀 Difficile        1  18.76  18.76  18.76
📈 Moyenne         17  25.99  18.83  32.41


,run_id,score,grade,cluster_label,anomaly_label
19,21,59.14,C,🏆 Elite,⚡ Exceptionnelle
12,14,39.03,D,⭐ Solide,Normal
1,3,32.41,D,📈 Moyenne,Normal
14,16,31.59,D,📈 Moyenne,Normal
3,5,29.88,D,📈 Moyenne,Normal
13,15,29.10,D,📈 Moyenne,Normal
16,18,28.58,D,📈 Moyenne,Normal
11,13,28.54,D,📈 Moyenne,Normal
9,11,28.33,D,📈 Moyenne,Normal
7,9,28.20,D,📈 Moyenne,Normal


In [32]:
# Remplace l'id par celui que tu veux inspecter
print_run_report(df_scored, run_id=df_scored.iloc[0]["run_id"])



╔══════════════════════════════════════════╗
║         RAPPORT DE RUN #21              ║
╠══════════════════════════════════════════╣
║  Score        : 59.1/100  [C]
║  Classement   : #1 / 20
║  Profil       : 🏆 Elite
║  Statut       : ⚡ Exceptionnelle
╠══════════════════════════════════════════╣
║  Victoire     : ❌
║  Stages       : 8
║  Rooms clear  : 98.6%
║  DPS proxy    : 0.55
║  Dégâts reçus : 4
║  Items passifs: 12 (qualité moy: 1.6)
║  Malédictions : 0
╚══════════════════════════════════════════╝
    


In [33]:
df_scored.to_csv("../data/run_scores.csv", index=False)
print("💾 Export terminé → data/run_scores.csv")


💾 Export terminé → data/run_scores.csv
